In [1]:
# Jupyter cell
# pip install pyserial

import serial
from typing import Dict, Optional

class BT1001L:
    """
    Longer BT100-1L peristaltic pump controller over RS485 serial.

    Protocol summary:
      - 1200 baud
      - 8 data bits
      - even parity
      - 1 stop bit
      - frame = E9 + addr + len + pdu + fcs
      - fcs = XOR(addr, len, *pdu)
      - byte stuffing in payload/fcs area:
          E8 -> E8 00
          E9 -> E8 01
    """

    FLAG = 0xE9
    ESC = 0xE8

    CMD_SET_SPEED = b"XL"
    CMD_READ_SPEED = b"DL"
    CMD_SET_FLOW  = b"WL"
    CMD_READ_FLOW = b"RL"
    CMD_CAL_FLOW  = b"CL"

    def __init__(self, port: str, pump_addr: int = 1, timeout: float = 1.0):
        if not (1 <= pump_addr <= 31):
            raise ValueError("pump_addr must be between 1 and 31")
        self.port = port
        self.pump_addr = pump_addr
        self.timeout = timeout
        self.ser: Optional[serial.Serial] = None

    def connect(self):
        self.ser = serial.Serial(
            port=self.port,
            baudrate=1200,
            bytesize=serial.EIGHTBITS,
            parity=serial.PARITY_EVEN,
            stopbits=serial.STOPBITS_ONE,
            timeout=self.timeout,
        )
        return self

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()

    def __enter__(self):
        return self.connect()

    def __exit__(self, exc_type, exc, tb):
        self.close()

    @staticmethod
    def _xor_checksum(addr: int, pdu: bytes) -> int:
        x = addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    @classmethod
    def _escape_bytes(cls, data: bytes) -> bytes:
        out = bytearray()
        for b in data:
            if b == cls.ESC:
                out.extend([cls.ESC, 0x00])
            elif b == cls.FLAG:
                out.extend([cls.ESC, 0x01])
            else:
                out.append(b)
        return bytes(out)

    @classmethod
    def _unescape_bytes(cls, data: bytes) -> bytes:
        out = bytearray()
        i = 0
        while i < len(data):
            b = data[i]
            if b == cls.ESC:
                if i + 1 >= len(data):
                    raise ValueError("Malformed escape sequence")
                nxt = data[i + 1]
                if nxt == 0x00:
                    out.append(cls.ESC)
                elif nxt == 0x01:
                    out.append(cls.FLAG)
                else:
                    raise ValueError(f"Invalid escaped byte sequence: E8 {nxt:02X}")
                i += 2
            else:
                out.append(b)
                i += 1
        return bytes(out)

    def _build_frame(self, pdu: bytes, addr: Optional[int] = None) -> bytes:
        if addr is None:
            addr = self.pump_addr
        fcs = self._xor_checksum(addr, pdu)
        body = bytes([addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([self.FLAG]) + self._escape_bytes(body)

    def _read_frame(self) -> bytes:
        if self.ser is None:
            raise RuntimeError("Serial port is not connected")

        # Wait for start flag
        while True:
            b = self.ser.read(1)
            if not b:
                raise TimeoutError("Timed out waiting for pump response")
            if b[0] == self.FLAG:
                break

        # Read the rest of the frame.
        # Response length is small; timeout terminates read.
        raw = bytearray()
        while True:
            b = self.ser.read(1)
            if not b:
                break
            raw.extend(b)

        if not raw:
            raise TimeoutError("Pump returned only start flag")

        frame = self._unescape_bytes(bytes(raw))
        if len(frame) < 3:
            raise ValueError(f"Response too short: {frame.hex(' ')}")

        addr = frame[0]
        pdu_len = frame[1]
        expected_total = 2 + pdu_len + 1  # addr + len + pdu + fcs

        if len(frame) != expected_total:
            raise ValueError(
                f"Unexpected response length: got {len(frame)}, expected {expected_total}. "
                f"Raw frame: {frame.hex(' ')}"
            )

        pdu = frame[2:2 + pdu_len]
        recv_fcs = frame[-1]
        calc_fcs = self._xor_checksum(addr, pdu)
        if recv_fcs != calc_fcs:
            raise ValueError(
                f"Checksum mismatch: recv=0x{recv_fcs:02X}, calc=0x{calc_fcs:02X}"
            )

        return frame

    def _send_command(self, pdu: bytes, expect_reply: bool = True, addr: Optional[int] = None) -> Optional[bytes]:
        if self.ser is None:
            raise RuntimeError("Serial port is not connected")

        frame = self._build_frame(pdu, addr=addr)
        self.ser.reset_input_buffer()
        self.ser.write(frame)
        self.ser.flush()

        if not expect_reply:
            return None

        resp = self._read_frame()
        resp_pdu = resp[2:-1]
        return resp_pdu

    @staticmethod
    def _state1(start: bool, prime: bool = False) -> int:
        val = 0
        if start:
            val |= 0b00000001
        if prime:
            val |= 0b00000010
        return val

    @staticmethod
    def _state2(cw: bool) -> int:
        return 0b00000001 if cw else 0

    def set_speed(self, rpm: float, start: bool = True, cw: bool = True, prime: bool = False) -> bytes:
        """
        Set speed in rpm. Pump uses 0.1 rpm units, max 100.0 rpm.
        """
        if not (0.0 <= rpm <= 100.0):
            raise ValueError("rpm must be between 0.0 and 100.0")

        speed_tenths = int(round(rpm * 10))
        speed_bytes = speed_tenths.to_bytes(2, byteorder="big", signed=False)

        pdu = (
            self.CMD_SET_SPEED +
            speed_bytes +
            bytes([self._state1(start, prime), self._state2(cw)])
        )
        return self._send_command(pdu, expect_reply=True)

    def read_speed(self) -> Dict[str, object]:
        """
        Read speed parameters from the pump.
        Response PDU:
          DL + show_speed(2) + state1 + state2
        """
        resp = self._send_command(self.CMD_READ_SPEED, expect_reply=True)
        if resp[:2] != self.CMD_READ_SPEED or len(resp) != 6:
            raise ValueError(f"Unexpected read_speed response: {resp.hex(' ')}")

        speed_tenths = int.from_bytes(resp[2:4], byteorder="big", signed=False)
        state1 = resp[4]
        state2 = resp[5]

        return {
            "rpm": speed_tenths / 10.0,
            "running": bool(state1 & 0b00000001),
            "prime": bool(state1 & 0b00000010),
            "cw": bool(state2 & 0b00000001),
            "raw_pdu": resp.hex(" "),
        }

    def set_flow(self, flow_ml_min: float, pump_head: int, tube_no: int,
                 start: bool = True, cw: bool = True, prime: bool = False) -> bytes:
        """
        Set flow in mL/min.
        Protocol uses nL/min as a 4-byte unsigned integer.
        """
        if flow_ml_min < 0:
            raise ValueError("flow_ml_min must be >= 0")

        flow_nl_min = int(round(flow_ml_min * 1_000_000))  # 1 mL = 1,000,000 nL
        if not (0 <= flow_nl_min <= 0xFFFFFFFF):
            raise ValueError("flow value out of 4-byte range")

        pdu = (
            self.CMD_SET_FLOW +
            flow_nl_min.to_bytes(4, byteorder="big", signed=False) +
            bytes([
                self._state1(start, prime),
                self._state2(cw),
                pump_head,
                tube_no
            ])
        )
        return self._send_command(pdu, expect_reply=True)

    def read_flow(self) -> Dict[str, object]:
        """
        Read flow parameters from the pump.
        Response PDU:
          RL + show_flow(4) + state1 + state2 + pump_head + tube_no
        """
        resp = self._send_command(self.CMD_READ_FLOW, expect_reply=True)
        if resp[:2] != self.CMD_READ_FLOW or len(resp) != 10:
            raise ValueError(f"Unexpected read_flow response: {resp.hex(' ')}")

        flow_nl_min = int.from_bytes(resp[2:6], byteorder="big", signed=False)
        state1 = resp[6]
        state2 = resp[7]
        pump_head = resp[8]
        tube_no = resp[9]

        return {
            "flow_nl_min": flow_nl_min,
            "flow_ml_min": flow_nl_min / 1_000_000,
            "running": bool(state1 & 0b00000001),
            "prime": bool(state1 & 0b00000010),
            "cw": bool(state2 & 0b00000001),
            "pump_head": pump_head,
            "tube_no": tube_no,
            "raw_pdu": resp.hex(" "),
        }

    def stop(self, cw: bool = True) -> bytes:
        """
        Stop the pump by writing speed 0 and start bit = 0.
        """
        pdu = self.CMD_SET_SPEED + (0).to_bytes(2, "big") + bytes([self._state1(False, False), self._state2(cw)])
        return self._send_command(pdu, expect_reply=True)

    def prime(self, cw: bool = True) -> bytes:
        """
        Prime at max speed using state1 bit1.
        """
        pdu = self.CMD_SET_SPEED + (0).to_bytes(2, "big") + bytes([self._state1(True, True), self._state2(cw)])
        return self._send_command(pdu, expect_reply=True)

In [2]:
# Change COM3 to your serial port.
# On Linux/macOS it may look like /dev/ttyUSB0 or /dev/tty.usbserial-xxxx

pump = BT1001L(port="COM10", pump_addr=1).connect()

# Example 1: set 20.0 rpm clockwise and start
resp = pump.set_speed(20.0, start=True, cw=True)
print("set_speed response:", resp.hex(" "))

# Read back current speed settings
print(pump.read_speed())

# Example 2: set 3.0 mL/min, DG(10 rollers)=2, tube 0.25 mm = 3, counter-clockwise
resp = pump.set_flow(
    flow_ml_min=3.0,
    pump_head=2,
    tube_no=3,
    start=True,
    cw=False
)
print("set_flow response:", resp.hex(" "))

# Read back current flow settings
print(pump.read_flow())

# Stop when done
pump.stop()
pump.close()

TimeoutError: Timed out waiting for pump response

In [1]:
import serial
import time

ser = serial.Serial(
    port="COM10",
    baudrate=1200,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_ONE,
    timeout=2.0,
)

# Protocol example on page 5:
# E9 01 06 58 4C 00 C8 01 01 DB
cmd = bytes.fromhex("E9 01 06 58 4C 00 C8 01 01 DB")

ser.reset_input_buffer()
ser.reset_output_buffer()

print("Writing:", cmd.hex(" "))
ser.write(cmd)
ser.flush()

time.sleep(0.5)
resp = ser.read_all()
print("Raw response:", resp.hex(" ") if resp else "<nothing>")

ser.close()

Writing: e9 01 06 58 4c 00 c8 01 01 db
Raw response: <nothing>


In [2]:
import serial
import time

class BT1001L:
    def __init__(self, port, addr=1):
        self.port = port
        self.addr = addr
        self.ser = None

    def open(self):
        if self.ser and self.ser.is_open:
            return

        self.ser = serial.Serial(
            port=self.port,
            baudrate=1200,
            bytesize=serial.EIGHTBITS,
            parity=serial.PARITY_EVEN,
            stopbits=serial.STOPBITS_ONE,
            timeout=2
        )
        print(f"Opened {self.port}")

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
            print(f"Closed {self.port}")

    def __enter__(self):
        self.open()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.close()

    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _frame(self, pdu):
        fcs = self._checksum(pdu)
        return bytes([0xE9, self.addr, len(pdu)]) + pdu + bytes([fcs])

    def send(self, pdu):
        frame = self._frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.write(frame)
        self.ser.flush()

        time.sleep(0.3)
        resp = self.ser.read_all()

        print("TX:", frame.hex(" "))
        print("RX:", resp.hex(" ") if resp else "<no response>")

        return resp

    def set_speed(self, rpm):
        speed = int(rpm * 10)  # 0.1 rpm unit
        speed_bytes = speed.to_bytes(2, "big")

        state1 = 0x01  # start
        state2 = 0x01  # clockwise

        pdu = b"XL" + speed_bytes + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL")

In [3]:
with BT1001L("COM10", addr=1) as pump:
    pump.set_speed(20.0)
    pump.read_speed()

Opened COM10
TX: e9 01 06 58 4c 00 c8 01 01 db
RX: <no response>
TX: e9 01 02 44 4c 0b
RX: <no response>
Closed COM10


In [1]:
import serial
import time

class BT1001L:
    def __init__(self, port, addr=1, timeout=1.5):
        self.port = port
        self.addr = addr
        self.timeout = timeout
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout,
            )
        return self

    def close(self):
        if self.ser is not None:
            try:
                if self.ser.is_open:
                    self.ser.close()
            finally:
                self.ser = None

    def __enter__(self):
        return self.open()

    def __exit__(self, exc_type, exc, tb):
        self.close()

    def _checksum(self, pdu: bytes) -> int:
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _build_frame(self, pdu: bytes) -> bytes:
        # No escaping needed for these two commands as formed here.
        return bytes([0xE9, self.addr, len(pdu)]) + pdu + bytes([self._checksum(pdu)])

    def send(self, pdu: bytes, wait_s: float = 0.4) -> bytes:
        if self.ser is None or not self.ser.is_open:
            raise RuntimeError("Serial port is not open")

        frame = self._build_frame(pdu)
        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()
        time.sleep(wait_s)
        resp = self.ser.read_all()

        print(f"ADDR {self.addr}")
        print("TX:", frame.hex(" "))
        print("RX:", resp.hex(" ") if resp else "<no response>")
        print()
        return resp

    def read_speed(self) -> bytes:
        return self.send(b"DL")

    def set_speed(self, rpm: float, start=True, cw=True) -> bytes:
        speed = int(round(rpm * 10))
        speed_bytes = speed.to_bytes(2, "big")
        state1 = 0x01 if start else 0x00
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + speed_bytes + bytes([state1, state2])
        return self.send(pdu)

In [2]:
for addr in range(1, 6):
    try:
        with BT1001L("COM10", addr=addr) as pump:
            pump.read_speed()
    except Exception as e:
        print(f"ADDR {addr} ERROR: {e}\n")

ADDR 1
TX: e9 01 02 44 4c 0b
RX: <no response>

ADDR 2
TX: e9 02 02 44 4c 08
RX: <no response>

ADDR 3
TX: e9 03 02 44 4c 09
RX: <no response>

ADDR 4
TX: e9 04 02 44 4c 0e
RX: <no response>

ADDR 5
TX: e9 05 02 44 4c 0f
RX: <no response>



In [33]:
with BT1001L("COM10", addr=1) as pump:
    pump.read_speed()

ADDR 1
TX: e9 01 02 44 4c 0b
RX: <no response>



In [4]:
with BT1001L("COM10", addr=1) as pump:
    pump.read_speed()

ADDR 1
TX: e9 01 02 44 4c 0b
RX: <no response>



In [5]:
# Jupyter cell
# pip install pyserial

import time
import serial


class BT1001L:
    """
    Longer BT100-1L controller over RS485.

    Based on:
    - Darwin Microfluidics Python tutorial
    - Longer RS485 protocol PDF
    """

    def __init__(self, port: str, addr: int = 1, timeout: float = 1.5):
        if not (1 <= addr <= 31):
            raise ValueError("addr must be between 1 and 31")
        self.port = port
        self.addr = addr
        self.timeout = timeout
        self.ser = None

    # ---------- connection handling ----------
    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout,
            )
            print(f"Opened {self.port}")
        return self

    def close(self):
        if self.ser is not None:
            try:
                if self.ser.is_open:
                    self.ser.close()
                    print(f"Closed {self.port}")
            finally:
                self.ser = None

    def __enter__(self):
        return self.open()

    def __exit__(self, exc_type, exc, tb):
        self.close()

    # ---------- protocol helpers ----------
    def _xor_checksum(self, pdu: bytes, addr: int | None = None) -> int:
        if addr is None:
            addr = self.addr
        x = addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _escape(self, data: bytes) -> bytes:
        """
        Protocol escaping:
        E8 -> E8 00
        E9 -> E8 01
        except the leading frame flag E9
        """
        out = bytearray()
        for b in data:
            if b == 0xE8:
                out.extend([0xE8, 0x00])
            elif b == 0xE9:
                out.extend([0xE8, 0x01])
            else:
                out.append(b)
        return bytes(out)

    def _unescape(self, data: bytes) -> bytes:
        out = bytearray()
        i = 0
        while i < len(data):
            b = data[i]
            if b == 0xE8:
                if i + 1 >= len(data):
                    raise ValueError("Malformed escape sequence")
                nxt = data[i + 1]
                if nxt == 0x00:
                    out.append(0xE8)
                elif nxt == 0x01:
                    out.append(0xE9)
                else:
                    raise ValueError(f"Invalid escape sequence: E8 {nxt:02X}")
                i += 2
            else:
                out.append(b)
                i += 1
        return bytes(out)

    def _build_frame(self, pdu: bytes, addr: int | None = None) -> bytes:
        if addr is None:
            addr = self.addr
        fcs = self._xor_checksum(pdu, addr)
        body = bytes([addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([0xE9]) + self._escape(body)

    def _read_raw(self, wait_s: float = 0.5) -> bytes:
        time.sleep(wait_s)
        return self.ser.read_all()

    def _parse_response(self, raw: bytes) -> dict | None:
        if not raw:
            return None
        if raw[0] != 0xE9:
            return {"raw": raw, "error": "Response does not start with E9"}

        body = self._unescape(raw[1:])
        if len(body) < 3:
            return {"raw": raw, "error": "Response too short"}

        addr = body[0]
        pdu_len = body[1]
        if len(body) != 2 + pdu_len + 1:
            return {"raw": raw, "error": "Unexpected response length", "body": body}

        pdu = body[2:2 + pdu_len]
        recv_fcs = body[-1]
        calc_fcs = self._xor_checksum(pdu, addr)

        return {
            "addr": addr,
            "pdu_len": pdu_len,
            "pdu": pdu,
            "recv_fcs": recv_fcs,
            "calc_fcs": calc_fcs,
            "checksum_ok": recv_fcs == calc_fcs,
            "raw": raw,
            "body": body,
        }

    def send_pdu(self, pdu: bytes, wait_s: float = 0.5, verbose: bool = True) -> dict | None:
        if self.ser is None or not self.ser.is_open:
            raise RuntimeError("Serial port is not open")

        frame = self._build_frame(pdu)
        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()

        raw = self._read_raw(wait_s=wait_s)

        if verbose:
            print("TX:", frame.hex(" "))
            print("RX:", raw.hex(" ") if raw else "<no response>")

        parsed = self._parse_response(raw)
        return parsed

    # ---------- command builders ----------
    @staticmethod
    def _state1(start: bool = True, prime: bool = False) -> int:
        val = 0
        if start:
            val |= 0x01
        if prime:
            val |= 0x02
        return val

    @staticmethod
    def _state2(cw: bool = True) -> int:
        return 0x01 if cw else 0x00

    def set_speed(self, rpm: float, start: bool = True, cw: bool = True, prime: bool = False,
                  wait_s: float = 0.5, verbose: bool = True):
        """
        Set rotation speed in rpm.
        Protocol unit is 0.1 rpm.
        """
        if not (0 <= rpm <= 100.0):
            raise ValueError("rpm must be between 0 and 100.0")

        speed_tenths = int(round(rpm * 10))
        pdu = (
            b"XL"
            + speed_tenths.to_bytes(2, "big")
            + bytes([self._state1(start, prime), self._state2(cw)])
        )
        return self.send_pdu(pdu, wait_s=wait_s, verbose=verbose)

    def read_speed(self, wait_s: float = 0.5, verbose: bool = True):
        return self.send_pdu(b"DL", wait_s=wait_s, verbose=verbose)

    def set_flow(self, flow_ml_min: float, pump_head: int, tube_no: int,
                 start: bool = True, cw: bool = True, prime: bool = False,
                 wait_s: float = 0.5, verbose: bool = True):
        """
        Set flow in mL/min.
        Protocol unit is nL/min.
        """
        if flow_ml_min < 0:
            raise ValueError("flow_ml_min must be >= 0")

        flow_nl_min = int(round(flow_ml_min * 1_000_000))
        pdu = (
            b"WL"
            + flow_nl_min.to_bytes(4, "big")
            + bytes([self._state1(start, prime), self._state2(cw), pump_head, tube_no])
        )
        return self.send_pdu(pdu, wait_s=wait_s, verbose=verbose)

    def read_flow(self, wait_s: float = 0.5, verbose: bool = True):
        return self.send_pdu(b"RL", wait_s=wait_s, verbose=verbose)

    def stop(self, cw: bool = True, wait_s: float = 0.5, verbose: bool = True):
        return self.set_speed(0.0, start=False, cw=cw, prime=False, wait_s=wait_s, verbose=verbose)

    # ---------- response decoders ----------
    @staticmethod
    def decode_speed_response(parsed: dict) -> dict:
        if parsed is None:
            return {"error": "No response"}
        if not parsed.get("checksum_ok", False):
            return {"error": "Checksum failed", **parsed}

        pdu = parsed["pdu"]
        if len(pdu) != 6 or pdu[:2] != b"DL":
            return {"error": "Unexpected speed response", **parsed}

        speed_tenths = int.from_bytes(pdu[2:4], "big")
        state1 = pdu[4]
        state2 = pdu[5]
        return {
            "rpm": speed_tenths / 10.0,
            "running": bool(state1 & 0x01),
            "prime": bool(state1 & 0x02),
            "cw": bool(state2 & 0x01),
            "raw_pdu": pdu.hex(" "),
        }

    @staticmethod
    def decode_flow_response(parsed: dict) -> dict:
        if parsed is None:
            return {"error": "No response"}
        if not parsed.get("checksum_ok", False):
            return {"error": "Checksum failed", **parsed}

        pdu = parsed["pdu"]
        if len(pdu) != 10 or pdu[:2] != b"RL":
            return {"error": "Unexpected flow response", **parsed}

        flow_nl_min = int.from_bytes(pdu[2:6], "big")
        state1 = pdu[6]
        state2 = pdu[7]
        pump_head = pdu[8]
        tube_no = pdu[9]

        return {
            "flow_nl_min": flow_nl_min,
            "flow_ml_min": flow_nl_min / 1_000_000,
            "running": bool(state1 & 0x01),
            "prime": bool(state1 & 0x02),
            "cw": bool(state2 & 0x01),
            "pump_head": pump_head,
            "tube_no": tube_no,
            "raw_pdu": pdu.hex(" "),
        }

In [7]:
# Read current speed
with BT1001L("COM10", addr=1) as pump:
    resp = pump.read_speed(wait_s=1.0)
    print(resp)
    print(BT1001L.decode_speed_response(resp))

Opened COM10
TX: e9 01 02 44 4c 0b
RX: <no response>
None
{'error': 'No response'}
Closed COM10


In [8]:
# Read current speed
with BT1001L("COM10", addr=1) as pump:
    resp = pump.read_speed(wait_s=1.0)
    print(resp)
    print(BT1001L.decode_speed_response(resp))

Opened COM10
TX: e9 01 02 44 4c 0b
RX: <no response>
None
{'error': 'No response'}
Closed COM10


In [9]:
import time
import serial

PORT = "COM10"

def open_port():
    return serial.Serial(
        port=PORT,
        baudrate=1200,
        bytesize=serial.EIGHTBITS,
        parity=serial.PARITY_EVEN,
        stopbits=serial.STOPBITS_ONE,
        timeout=1.5,
    )

def xor_checksum(addr: int, pdu: bytes) -> int:
    x = addr ^ len(pdu)
    for b in pdu:
        x ^= b
    return x

def escape_payload(data: bytes) -> bytes:
    out = bytearray()
    for b in data:
        if b == 0xE8:
            out.extend([0xE8, 0x00])
        elif b == 0xE9:
            out.extend([0xE8, 0x01])
        else:
            out.append(b)
    return bytes(out)

def build_frame(addr: int, pdu: bytes) -> bytes:
    fcs = xor_checksum(addr, pdu)
    body = bytes([addr, len(pdu)]) + pdu + bytes([fcs])
    return bytes([0xE9]) + escape_payload(body)

def send_raw(addr: int, pdu: bytes, wait_s: float = 0.7):
    frame = build_frame(addr, pdu)
    with open_port() as ser:
        ser.reset_input_buffer()
        ser.reset_output_buffer()
        ser.write(frame)
        ser.flush()
        time.sleep(wait_s)
        resp = ser.read_all()

    print(f"ADDR {addr}")
    print("TX:", frame.hex(" "))
    print("RX:", resp.hex(" ") if resp else "<no response>")
    return resp

In [10]:
with open_port() as ser:
    ser.reset_input_buffer()
    ser.write(b"ABC")
    ser.flush()
    time.sleep(0.5)
    print(ser.read_all())

b''


In [11]:
# Broadcast start at 20.0 rpm clockwise
pdu = b"XL" + bytes.fromhex("00 C8") + bytes([0x01, 0x01])
send_raw(0x1F, pdu)

ADDR 31
TX: e9 1f 06 58 4c 00 c8 01 01 c5
RX: <no response>


b''

In [12]:
# Broadcast stop
pdu = b"XL" + bytes.fromhex("00 00") + bytes([0x00, 0x01])
send_raw(0x1F, pdu)

ADDR 31
TX: e9 1f 06 58 4c 00 00 00 01 0c
RX: <no response>


b''

In [13]:
send_raw(1, b"DL")

ADDR 1
TX: e9 01 02 44 4c 0b
RX: <no response>


b''

start

In [14]:
pdu = b"XL" + bytes.fromhex("00 C8") + bytes([0x01, 0x01])
send_raw(0x1F, pdu)

ADDR 31
TX: e9 1f 06 58 4c 00 c8 01 01 c5
RX: <no response>


b''

stop

In [ ]:
pdu = b"XL" + bytes.fromhex("00 00") + bytes([0x00, 0x01])
send_raw(0x1F, pdu)

In [ ]:
import time
import serial

PORT = "COM10"

def open_port():
    return serial.Serial(
        port=PORT,
        baudrate=1200,
        bytesize=serial.EIGHTBITS,
        parity=serial.PARITY_EVEN,
        stopbits=serial.STOPBITS_ONE,
        timeout=1
    )

def xor_checksum(addr, pdu):
    x = addr ^ len(pdu)
    for b in pdu:
        x ^= b
    return x

def build_frame(addr, pdu):
    fcs = xor_checksum(addr, pdu)
    return bytes([0xE9, addr, len(pdu)]) + pdu + bytes([fcs])

def send(addr, pdu):
    frame = build_frame(addr, pdu)
    ser.write(frame)
    ser.flush()
    print("TX:", frame.hex(" "))

# Broadcast address
ADDR = 0x1F

# Commands
START = b"XL" + bytes.fromhex("00 C8") + bytes([0x01, 0x01])  # 20 rpm CW
STOP  = b"XL" + bytes.fromhex("00 00") + bytes([0x00, 0x01])  # stop

ser = open_port()

print("=== STARTING LOOP ===")
print("Swap wires while watching pump...")

try:
    while True:
        print("\n>>> START")
        send(ADDR, START)
        time.sleep(2)

        print(">>> STOP")
        send(ADDR, STOP)
        time.sleep(2)

except KeyboardInterrupt:
    print("\nStopped by user")

finally:
    ser.close()
    print("COM port closed")

=== STARTING LOOP ===
Swap wires while watching pump...

>>> START
TX: e9 1f 06 58 4c 00 c8 01 01 c5
>>> STOP
TX: e9 1f 06 58 4c 00 00 00 01 0c

>>> START
TX: e9 1f 06 58 4c 00 c8 01 01 c5
>>> STOP
TX: e9 1f 06 58 4c 00 00 00 01 0c

>>> START
TX: e9 1f 06 58 4c 00 c8 01 01 c5
>>> STOP
TX: e9 1f 06 58 4c 00 00 00 01 0c

>>> START
TX: e9 1f 06 58 4c 00 c8 01 01 c5
>>> STOP
TX: e9 1f 06 58 4c 00 00 00 01 0c

>>> START
TX: e9 1f 06 58 4c 00 c8 01 01 c5


In [16]:
import time
import serial

class BT1001L:
    def __init__(self, port, addr=1, timeout=1.5):
        self.port = port
        self.addr = addr
        self.timeout = timeout
        self.ser = None

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout,
            )
            print(f"Opened {self.port}")
        return self

    def close(self):
        if self.ser is not None:
            try:
                if self.ser.is_open:
                    self.ser.close()
                    print(f"Closed {self.port}")
            finally:
                self.ser = None

    def __enter__(self):
        return self.open()

    def __exit__(self, exc_type, exc, tb):
        self.close()

    def _checksum(self, pdu: bytes, addr=None) -> int:
        if addr is None:
            addr = self.addr
        x = addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _escape(self, data: bytes) -> bytes:
        out = bytearray()
        for b in data:
            if b == 0xE8:
                out.extend([0xE8, 0x00])
            elif b == 0xE9:
                out.extend([0xE8, 0x01])
            else:
                out.append(b)
        return bytes(out)

    def _unescape(self, data: bytes) -> bytes:
        out = bytearray()
        i = 0
        while i < len(data):
            if data[i] == 0xE8:
                if i + 1 >= len(data):
                    raise ValueError("Bad escape sequence")
                if data[i + 1] == 0x00:
                    out.append(0xE8)
                elif data[i + 1] == 0x01:
                    out.append(0xE9)
                else:
                    raise ValueError("Bad escape sequence")
                i += 2
            else:
                out.append(data[i])
                i += 1
        return bytes(out)

    def _build_frame(self, pdu: bytes, addr=None) -> bytes:
        if addr is None:
            addr = self.addr
        fcs = self._checksum(pdu, addr)
        body = bytes([addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([0xE9]) + self._escape(body)

    def send_pdu(self, pdu: bytes, wait_s=0.5):
        frame = self._build_frame(pdu)
        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()
        self.ser.write(frame)
        self.ser.flush()
        time.sleep(wait_s)
        raw = self.ser.read_all()

        print("TX:", frame.hex(" "))
        print("RX:", raw.hex(" ") if raw else "<no response>")
        return raw

    def set_speed(self, rpm: float, start=True, cw=True, prime=False, wait_s=0.5):
        speed = int(round(rpm * 10))
        state1 = (0x01 if start else 0x00) | (0x02 if prime else 0x00)
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + speed.to_bytes(2, "big") + bytes([state1, state2])
        return self.send_pdu(pdu, wait_s=wait_s)

    def read_speed(self, wait_s=0.8):
        return self.send_pdu(b"DL", wait_s=wait_s)

    def stop(self, wait_s=0.5):
        return self.set_speed(0.0, start=False, cw=True, wait_s=wait_s)

In [17]:
with BT1001L("COM10", addr=1) as pump:
    pump.read_speed(wait_s=1.0)

Opened COM10
TX: e9 01 02 44 4c 0b
RX: <no response>
Closed COM10


In [19]:
for addr in range(1, 6):
    print(f"\n--- testing addr {addr} ---")
    with BT1001L("COM10", addr=addr) as pump:
        pump.read_speed(wait_s=1.0)


--- testing addr 1 ---
Opened COM10
TX: e9 01 02 44 4c 0b
RX: <no response>
Closed COM10

--- testing addr 2 ---
Opened COM10
TX: e9 02 02 44 4c 08
RX: <no response>
Closed COM10

--- testing addr 3 ---
Opened COM10
TX: e9 03 02 44 4c 09
RX: <no response>
Closed COM10

--- testing addr 4 ---
Opened COM10
TX: e9 04 02 44 4c 0e
RX: <no response>
Closed COM10

--- testing addr 5 ---
Opened COM10
TX: e9 05 02 44 4c 0f
RX: <no response>
Closed COM10


In [20]:
with BT1001L("COM10", addr=31) as pump:
    pump.set_speed(20.0, start=True, cw=True)
    time.sleep(3)
    pump.stop()

Opened COM10
TX: e9 1f 06 58 4c 00 c8 01 01 c5
RX: <no response>
TX: e9 1f 06 58 4c 00 00 00 01 0c
RX: <no response>
Closed COM10


In [21]:
with BT1001L("COM10", addr=1) as pump:
    pump.set_speed(20.0, start=True, cw=True, wait_s=1.0)
    pump.read_speed(wait_s=1.0)

Opened COM10
TX: e9 01 06 58 4c 00 c8 01 01 db
RX: <no response>
TX: e9 01 02 44 4c 0b
RX: <no response>
Closed COM10


In [22]:
with BT1001L("COM10", addr=1) as pump:
    pump.read_speed(wait_s=1.0)

Opened COM10
TX: e9 01 02 44 4c 0b
RX: <no response>
Closed COM10


In [35]:
for addr in range(1, 6):
    print(f"\n--- testing addr {addr} ---")
    with BT1001L("COM10", addr=addr) as pump:
        pump.read_speed(wait_s=1.0)


--- testing addr 1 ---
Opened COM10
TX: e9 01 02 44 4c 0b
RX: e9 01 06 44 4c 00 01 00 01 0f
Closed COM10

--- testing addr 2 ---
Opened COM10
TX: e9 02 02 44 4c 08
RX: <no response>
Closed COM10

--- testing addr 3 ---
Opened COM10
TX: e9 03 02 44 4c 09
RX: <no response>
Closed COM10

--- testing addr 4 ---
Opened COM10
TX: e9 04 02 44 4c 0e
RX: <no response>
Closed COM10

--- testing addr 5 ---
Opened COM10
TX: e9 05 02 44 4c 0f
RX: <no response>
Closed COM10


In [34]:
with BT1001L("COM10", addr=31) as pump:
    pump.set_speed(20.0, start=True, cw=True, wait_s=0.5)
    time.sleep(3)
    pump.stop(wait_s=0.5)

Opened COM10
TX: e9 1f 06 58 4c 00 c8 01 01 c5
RX: <no response>
TX: e9 1f 06 58 4c 00 00 00 01 0c
RX: <no response>
Closed COM10


In [33]:
import time

with BT1001L("COM10", addr=31) as pump:
    pump.set_speed(30.0, start=True, cw=True, wait_s=0.5)
    time.sleep(3)
    pump.stop(wait_s=5)

Opened COM10
TX: e9 1f 06 58 4c 01 2c 01 01 20
RX: <no response>
TX: e9 1f 06 58 4c 00 00 00 01 0c
RX: <no response>
Closed COM10


In [37]:
import time

with BT1001L("COM10", addr=31) as pump:
    pump.set_speed(20.0, start=True, cw=True, wait_s=0.5)
    time.sleep(3)
    pump.stop(wait_s=0.5)

Opened COM10
TX: e9 1f 06 58 4c 00 c8 01 01 c5
RX: <no response>
TX: e9 1f 06 58 4c 00 00 00 01 0c
RX: <no response>
Closed COM10


In [38]:
import time
import serial


class BT1001L:
    def __init__(self, port, addr=31, timeout=1.0):
        self.port = port
        self.addr = addr
        self.timeout = timeout
        self.ser = None

    # ---------- connection ----------
    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=self.timeout,
            )
            print(f"Opened {self.port}")
        return self

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
            print(f"Closed {self.port}")

    def __enter__(self):
        return self.open()

    def __exit__(self, exc_type, exc, tb):
        self.close()

    # ---------- protocol ----------
    def _checksum(self, pdu):
        x = self.addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    def _build_frame(self, pdu):
        return bytes([0xE9, self.addr, len(pdu)]) + pdu + bytes([self._checksum(pdu)])

    def send(self, pdu, wait_s=0.3):
        frame = self._build_frame(pdu)

        self.ser.reset_input_buffer()
        self.ser.reset_output_buffer()

        self.ser.write(frame)
        self.ser.flush()

        time.sleep(wait_s)
        resp = self.ser.read_all()

        print("TX:", frame.hex(" "))
        print("RX:", resp.hex(" ") if resp else "<no response>")

        return resp

    # ---------- commands ----------
    def set_speed(self, rpm, start=True, cw=True):
        speed = int(rpm * 10)  # 0.1 rpm unit
        speed_bytes = speed.to_bytes(2, "big")

        state1 = 0x01 if start else 0x00
        state2 = 0x01 if cw else 0x00

        pdu = b"XL" + speed_bytes + bytes([state1, state2])
        return self.send(pdu)

    def stop(self):
        return self.set_speed(0.0, start=False)

    def read_speed(self):
        return self.send(b"DL")

In [39]:
import time

with BT1001L("COM10", addr=31) as pump:
    pump.set_speed(20.0)
    time.sleep(5)
    pump.stop()

Opened COM10
TX: e9 1f 06 58 4c 00 c8 01 01 c5
RX: <no response>
TX: e9 1f 06 58 4c 00 00 00 01 0c
RX: <no response>
Closed COM10


In [40]:
with BT1001L("COM10", addr=1) as pump:
    pump.read_speed()

Opened COM10
TX: e9 01 02 44 4c 0b
RX: e9 01 06 44 4c 00 01 00 01 0f
Closed COM10
